### ==============================================================================
# 🚀 SPACESHIP TITANIC - END-TO-END WINNING PIPELINE
# Auteur : Chouaib Mouid
# Score Leaderboard Public : 0.81014
# ==============================================================================

In [ ]:
#Importation des prérequis.

import pandas as pd
import numpy as np
import warnings
import lightgbm as lgb
import xgboost as xgb
import catboost as cb
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, roc_auc_score

warnings.filterwarnings('ignore')

print("="*60)
print(" 🚀 DÉMARRAGE DU PIPELINE DE MACHINE LEARNING ")
print("="*60)

print("\n[1/5] Chargement des jeux d'entraînement et de test...")
train_df = pd.read_csv('/content/train.csv')
test_df = pd.read_csv('/content/test.csv')

print(f" -> Train shape : {train_df.shape}")
print(f" -> Test shape  : {test_df.shape}")

 🚀 DÉMARRAGE DU PIPELINE DE MACHINE LEARNING 

[1/5] Chargement des jeux d'entraînement et de test...
 -> Train shape : (8693, 14)
 -> Test shape  : (4277, 13)


### 2. PIPELINE DE FEATURE ENGINEERING


In [98]:
def engineer_features(df_in):
    df = df_in.copy()
    pass_id = df['PassengerId'].copy() if 'PassengerId' in df.columns else None

    # A. Extraction des groupes de passagers
    if 'PassengerId' in df.columns:
        df['PassengerID_Groupe'] = df['PassengerId'].str.split('_').str[0]
        df['Groupe_size'] = df['PassengerID_Groupe'].map(df['PassengerID_Groupe'].value_counts())
        df['Is_alone'] = (df['Groupe_size'] == 1).astype(int)

    # B. Extraction de la famille via le nom
    if 'Name' in df.columns:
        df['Surname'] = df['Name'].str.split().str[-1]
        df['Family_size'] = df['Surname'].map(df['Surname'].value_counts())
    df = df.drop(columns=['PassengerId', 'Name', 'Surname', 'PassengerID_Groupe'], errors='ignore')

    # C. Localisation spatiale (Cabine)
    if 'Cabin' in df.columns:
        df['Deck'] = df['Cabin'].str.split('/').str[0]
        df['Num'] = pd.to_numeric(df['Cabin'].str.split('/').str[1])
        df['Side'] = df['Cabin'].str.split('/').str[2]
        df = df.drop(columns=['Cabin'], errors='ignore')

    # D. Gestion des dépenses et comportements financiers
    expense_cols = ['RoomService', 'FoodCourt', 'ShoppingMall', 'Spa', 'VRDeck']
    for col in expense_cols:
        if col in df.columns:
            df[col] = df[col].fillna(0)

    if all(col in df.columns for col in expense_cols):
        df['Total_expenses'] = df[expense_cols].sum(axis=1)
        df['Has_Paid_Expenses'] = (df['Total_expenses'] > 0).astype(int)

    # E. Démographie
    if 'Age' in df.columns:
        df['Is-child'] = (df['Age'] < 18).astype(int)

    # F. Conversion des colonnes object/bool en category pour les modèles arborescents
    for col in df.select_dtypes(include=['object', 'bool']).columns:
        df[col] = df[col].astype('category')

    return df, pass_id

print("\n[2/5] Application du Feature Engineering...")
y = train_df['Transported'].astype(int)
train_features = train_df.drop(columns=['Transported'])

X, _ = engineer_features(train_features)
X_test, pass_id = engineer_features(test_df)

print(f" -> X (Train) transformé : {X.shape}")
print(f" -> X_test transformé     : {X_test.shape}")


[2/5] Application du Feature Engineering...
 -> X (Train) transformé : (8693, 19)
 -> X_test transformé     : (4277, 19)


### 3. VALIDATION CROISÉE STRATIFIÉE (5-FOLDS) & ÉVALUATION OOF


In [104]:
print("\n[3/5] Lancement de la validation croisée...")
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

oof_lgb = np.zeros(len(X))
oof_xgb = np.zeros(len(X))
oof_cb = np.zeros(len(X))

for fold, (train_idx, val_idx) in enumerate(skf.split(X, y)):
    X_train, y_train = X.iloc[train_idx], y.iloc[train_idx]
    X_val, y_val = X.iloc[val_idx], y.iloc[val_idx]

    # --- LightGBM ---
    model_lgb = lgb.LGBMClassifier(
        random_state=42, n_estimators=910, learning_rate=0.025,
        num_leaves=29, max_depth=5, min_child_samples=34,
        subsample=0.90, colsample_bytree=0.95,
        reg_alpha=0.156, reg_lambda=0.516, verbose=-1
    )
    model_lgb.fit(X_train, y_train)
    oof_lgb[val_idx] = model_lgb.predict_proba(X_val)[:, 1]

    # --- XGBoost ---
    X_tr_xgb, X_vl_xgb = X_train.copy(), X_val.copy()
    for df_t in [X_tr_xgb, X_vl_xgb]:
        for col in df_t.columns:
            if df_t[col].dtype.name == 'category':
                df_t[col] = df_t[col].astype(str).astype('category')

    model_xgb = xgb.XGBClassifier(
        random_state=42, n_estimators=959, learning_rate=0.010797,
        max_depth=6, min_child_weight=1, subsample=0.7549,
        colsample_bytree=0.7238, reg_alpha=0.9809, reg_lambda=0.5253,
        enable_categorical=True, tree_method='hist', eval_metric='logloss', verbosity=0
    )
    model_xgb.fit(X_tr_xgb, y_train)
    oof_xgb[val_idx] = model_xgb.predict_proba(X_vl_xgb)[:, 1]

    # --- CatBoost ---
    cat_cols = list(X.select_dtypes(include=['category', 'object']).columns)
    X_tr_cb, X_vl_cb = X_train.copy(), X_val.copy()
    for df_t in [X_tr_cb, X_vl_cb]:
        for col in cat_cols:
            if "Missing" not in df_t[col].cat.categories:
                df_t[col] = df_t[col].cat.add_categories(["Missing"])
            df_t[col] = df_t[col].fillna("Missing").astype(str)

    model_cb = cb.CatBoostClassifier(
        random_state=42, iterations=1023, learning_rate=0.028297,
        depth=7, l2_leaf_reg=0.07938, bagging_temperature=0.72408,
        cat_features=cat_cols, verbose=0
    )
    model_cb.fit(X_tr_cb, y_train)
    oof_cb[val_idx] = model_cb.predict_proba(X_vl_cb)[:, 1]

# Calcul et affichage du score local Out-Of-Fold
weights = [0.4187, 0.3020, 0.2793]
final_oof_probs = weights[0] * oof_lgb + weights[1] * oof_xgb + weights[2] * oof_cb
best_thresh = 0.52
oof_preds = (final_oof_probs >= best_thresh).astype(int)

local_accuracy = accuracy_score(y, oof_preds)
local_auc = roc_auc_score(y, final_oof_probs)

print(f"\n🎯 [RÉSULTAT LOCAL] Accuracy OOF : {local_accuracy:.5f} | AUC : {local_auc:.5f}")


[3/5] Lancement de la validation croisée...

🎯 [RÉSULTAT LOCAL] Accuracy OOF : 0.81077 | AUC : 0.90529


### 4. ENTRAÎNEMENT FINAL SUR TOUT LE DATASET (TRAIN COMPLET)

In [ ]:
print("\n[4/5] Entraînement final des 3 modèles sur l'intégralité du train set...")

# LightGBM Full
model_lgb_full = lgb.LGBMClassifier(
    random_state=42, n_estimators=910, learning_rate=0.025,
    num_leaves=29, max_depth=5, min_child_samples=34,
    subsample=0.90, colsample_bytree=0.95, reg_alpha=0.156, reg_lambda=0.516, verbose=-1
)
model_lgb_full.fit(X, y)
lgb_test_preds = model_lgb_full.predict_proba(X_test)[:, 1]

# XGBoost Full
X_xgb_full = X.copy()
X_test_xgb = X_test.copy()
for df_t in [X_xgb_full, X_test_xgb]:
    for col in df_t.columns:
        if df_t[col].dtype.name == 'category':
            df_t[col] = df_t[col].astype(str).astype('category')

model_xgb_full = xgb.XGBClassifier(
    random_state=42, n_estimators=959, learning_rate=0.010797,
    max_depth=6, min_child_weight=1, subsample=0.7549,
    colsample_bytree=0.7238, reg_alpha=0.9809, reg_lambda=0.5253,
    enable_categorical=True, tree_method='hist', eval_metric='logloss', verbosity=0
)
model_xgb_full.fit(X_xgb_full, y)
xgb_test_preds = model_xgb_full.predict_proba(X_test_xgb)[:, 1]

# CatBoost Full
X_cb_full = X.copy()
X_test_cb = X_test.copy()
cat_cols = list(X.select_dtypes(include=['category', 'object']).columns)
for df_t in [X_cb_full, X_test_cb]:
    for col in cat_cols:
        if "Missing" not in df_t[col].cat.categories:
            df_t[col] = df_t[col].cat.add_categories(["Missing"])
        df_t[col] = df_t[col].fillna("Missing").astype(str)

model_cb_full = cb.CatBoostClassifier(
    random_state=42, iterations=1023, learning_rate=0.028297,
    depth=7, l2_leaf_reg=0.07938, bagging_temperature=0.72408, cat_features=cat_cols, verbose=0
)
model_cb_full.fit(X_cb_full, y)
cb_test_preds = model_cb_full.predict_proba(X_test_cb)[:, 1]


# ==============================================================================
# 5. BLENDING PONDÉRÉ & EXPORTATION DU FICHIER DE SOUMISSION
# ==============================================================================
print("\n[5/5] Application du blending pondéré et génération du fichier final...")
final_test_probs = weights[0] * lgb_test_preds + weights[1] * xgb_test_preds + weights[2] * cb_test_preds
final_predictions = final_test_probs >= best_thresh

submission = pd.DataFrame({
    'PassengerId': pass_id,
    'Transported': final_predictions.astype(bool)
})

output_filename = 'submission_spaceship_titanic_final.csv'
submission.to_csv(output_filename, index=False)

print(f"✅ Pipeline exécuté avec succès ! Fichier '{output_filename}' généré.")
print("="*60)